# పాఠం 08 - బహు-ఏజెంట్ డిజయిన్ నమూనా


## సెటప్


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ఎందుకు బహుళ ఏజెంట్ సిస్టమ్‌లు?

ప్రయాణ నిర్ధారణ వంటి వాస్తవ ప్రపంచ పనులు అనేక రకాల నైపుణ్యాలను అవసరమవుతాయి — సరుకు సరఫరా, స్థానిక జ్ఞానం, బడ్జెట్ నిర్వహణ, మరియు మరిన్ని. ఒకే ఏజెంట్ సమస్తం నిర్వహించడానికి యత్నించడం త్వరితంగా కఠినంగా మారుతుంది.

బహుళ ఏజెంట్ సిస్టమ్‌లు దీన్ని **ప్రత్యేకత** ద్వారా పరిష్కరిస్తాయి: ప్రతి ఏజెంట్ ఒక నైపుణ్య రంగంపై దృష్టి సారించి, సాధారణ వ్యక్తి కన్నా ఉన్నతమైన ఫలితాలను تولید చేస్తుంది. అవి కూడా **విస్తరణ సామర్థ్యం**ను మెరుగుపరిచాయి — మీరు కొత్త ఏజెంట్లను (ఉదాహరణకు, విమాన ప్రత్యేకజ్ఞుడు, రెస్టారెంట్ విమర్శకుడు) ప్రస్తుత పని ప్రవాహాన్ని తిరిగి రాసుకోకుండా జోడించవచ్చు. ఏజెంట్లు ఒక నిర్మిత pipeline ద్వారా కలిసె పని చేస్తాయి, ఒకరి నుంచి మరొకరికి సందర్భాన్ని փոխանցిస్తూ.


## ప్రత్యేక ఏజెంట్లను సృష్టించడం


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## ఒక క్రమబద్ధమైన వర్క్‌ఫ్లో తయారు చెయ్యడం

`WorkflowBuilder` మీరు ఏజెంట్స్‌ని ఒక దిశాబద్ద గ్రాఫ్‌లో వైర్ చేయడానికి అనుమతిస్తుంది. ఇక్కడ మనం ఒక సాదాసీదీ రెండు దశల పైప్లైన్‌ను సృష్టిస్తాము: **TravelPlanner** ప్రయాణ షెడ్యూల్‌ను తయారు చేస్తుంది, తర్వాత **TravelConcierge** దాన్ని సమీక్షించి మెరుగుపరుస్తుంది.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## వర్క్‌ఫ్లోకు మరిన్ని ఏజెంట్లను జోడించడం

బహుళ ఏజెంట్ నమూనాతో ఉన్న పెద్ద ప్రయోజనాల్లో ఒకటి అది ఎంత సులభంగా విస్తరించగలిగినదని ఉంది. క్రింద మేము ఒక **BudgetReviewer** ఏజెంట్‌ని జోడిస్తున్నాము, ఇది ప్రణాళికను ప్రయాణికుడు బడ్జెట్‌కు సరిపోల్చి పరిశీలిస్తుంది, ఖర్చులు పరిమితిని దాటే అవకాశమున్న అంశాలను గుర్తిస్తుంది, మరియు డబ్బు ఆదా చేసే ప్రత్యామ్నాయాలను సూచిస్తుంది. వర్క్‌ఫ్లో ఇప్పుడు మూడు ఏజెంట్లను వరుసగా నడుపుతుంది:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Summary

ఈ పాఠంలో మీరు ఎలా చేసుకోవాలో నేర్చుకున్నారు:

1. **ప్రత్యేక నాయకులను తయారు చేయండి** — ప్రతి ఒకరి స్పష్టమైన పాత్రతో (యాజమాన్యం, కన్సెర్జ్, బడ్జెట్ సమీక్ష).
2. **నాయకులను వారి వరుస పని ప్రవాహంలో కనెక్ట్ చేయండి** `WorkflowBuilder` మరియు `add_edge` ఉపయోగించి.
3. **మల్టీ-ఎజెంట్ పైప్లైన్ నుండి అవుట్పుట్ స్ట్రీమ్ చేయండి**, ఏ ఏజెంట్ మాట్లాడుతున్నదో ట్రాక్ చేస్తూ.
4. **పని ప్రవాహాన్ని పొడిగించండి** కొత్త ప్రభుత్వులను చేర్చడం ద్వారా, ఉన్నవాటిని మార్చకుండా.

మల్టీ-ఎజెంట్ డిజైన్ నమూనా ప్రతి ఏజెంట్ ను సింపుల్‌గా ఉంచుతుంది, కానీ ఒక్క ఏజెంట్ సాధించగలిగినదానికన్నా ధనికంగా, మరింత లోతుగా సమీక్షించిన ఫలితాలు ఉత్పత్తి చేస్తుంది.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**డిస్క్లెయిమర్**:
ఈ పత్రాన్ని AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడ్డది. మేము సరిగ్గా ఉండటానికి శ్రమిస్తామని గొప్ప సాయం తీసుకున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాల్లో తప్పులు లేదా అసంపూర్ణతలు ఉండవచ్చని దయచేసి గమనించండి. మూల పత్రాన్ని దాని స్థానిక భాషలో అధికారికమైన వనరుగా పరిగణించాలి. అత్యవసర సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదం కొరత మంచి. ఈ అనువాదాన్ని ఉపయోగించడం వల్ల వచ్చే ఏవైనా అవగాహన లోపాలు లేదా తప్పుల కోసం మేము బాధ్యులేం కśmy.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
